[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_32_A2A_Protocol_Agent_To_Agent_Messaging.ipynb)

# Lesson 32 — A2A Protocol & Agent-to-Agent Messaging

**Phase 4 · Track 2 — Multi-Agent Coordination · Lesson 1 of 5**

Today we cross a real architectural boundary. Up through Lesson 31, every agent we built lived in **one Python process**: the orchestrator imported the critic, called `critic(draft)`, got a return value. That works until it doesn't — until the critic needs a different model, a different repo owner, a different deploy cadence, or simply too much memory to share a runtime with the orchestrator.

**Agent-to-Agent (A2A)** is Google's open protocol (announced April 2025) for letting agents in *different processes, different machines, different organizations* discover each other and collaborate over plain HTTP. Think of it as the analogue of MCP, but one layer up:

| Protocol | Connects | Direction | Lesson |
|---|---|---|---|
| **MCP** | Tool ↔ Agent | An agent calls tools an MCP server exposes | L14 |
| **A2A** | Agent ↔ Agent | An agent delegates a *task* to another agent | L32 (today) |

They compose: an A2A agent can have MCP tools wired underneath it. We will build that exact composition today.

## What you'll build by the end of this notebook

1. Two FastAPI services — a **Researcher** agent and a **Critic** agent — each exposing the four standard A2A endpoints (`/.well-known/agent.json`, `POST /tasks/send`, `GET /tasks/{id}`, `GET /tasks/{id}/stream`).
2. A small `A2AClient` that *discovers* a peer agent from its **AgentCard**, sends a `Task`, polls until terminal state, and pulls the final `Artifact`.
3. A streaming variant that consumes the critic's reasoning over **Server-Sent Events**.
4. A port of AutoResearcher's L23 critic-loop — instead of `critic_in_process(draft)`, the orchestrator now calls `critic_via_a2a(draft)` and the critic could in principle run in a different repo / different team's deploy.

**Prerequisite mental model:** Lessons 4 (ReAct loop), 6 (multi-agent), 11 (LangGraph), 14 (MCP), 16 (FastAPI deployment), 19 (SSE streaming), 23 (AutoResearcher v1). We are bolting the *network* onto multi-agent.


## Why A2A exists — the failure mode of single-agent ReAct

From Lesson 4 onward we built one pattern over and over: a single ReAct loop that owns a tool registry, picks tools, observes results, and iterates. We saw the cracks already:

- **Context-window saturation.** Every tool result, every critic comment, every prior turn lives in the same conversation. By Lesson 23 the orchestrator's prompt was already several KB.
- **Tool sprawl.** As we added critic, citation-checker, calibrator, moderator — each became another tool the orchestrator had to *know about and choose between*. Single-agent attention gets thinner every time.
- **No parallelism.** ReAct is fundamentally sequential — one thought, one tool, one observation. Fanning out three searches in parallel (Lesson 6) was already awkward.
- **No isolation of failure.** If the critic crashes, the orchestrator crashes. No bulkheads.
- **No team boundaries.** A *different team* cannot ship the critic without also shipping the orchestrator.

Multi-agent systems (L6) fixed the conceptual problem but kept everything in-process. A2A fixes the *network* problem: agents become services, addressable by URL, discoverable by AgentCard, with a typed lifecycle.

### The mental shift

Think of A2A as **HTTP + a tiny agreed-upon schema** for what one agent says to another. There's no magic. Once you internalize the four endpoints and three data shapes, the rest is plumbing you've seen since Lesson 16.


## A2A vs MCP — they're not competing, they compose

Newbies often ask: *if I have MCP, why do I need A2A?* Different problems:

```
                   ┌───────────────────────────┐
                   │       User / Client       │
                   └─────────────┬─────────────┘
                                 │ A2A (Task)
             ┌───────────────────▼───────────────────┐
             │            Researcher Agent           │
             │   (its own LLM, its own ReAct loop)   │
             └────┬─────────────────────────┬────────┘
                  │ MCP                     │ A2A (Task)
        ┌─────────▼─────────┐     ┌─────────▼─────────┐
        │ search_web MCP    │     │   Critic Agent    │
        │  (tool server)    │     │ (own LLM/policy)  │
        └───────────────────┘     └───────────────────┘
```

- **MCP** turns a *tool* into a network service. The MCP server is *not* an agent — it can't decide; it just exposes capabilities (`search_web`, `read_file`).
- **A2A** turns an *agent* into a network service. The A2A server *can* decide — it runs its own ReAct loop, calls its own tools (possibly via MCP), and returns a *task result*.

Rule of thumb: if the remote thing **takes a goal and decides how to achieve it**, it's an A2A agent. If it **takes a function call and returns a value**, it's an MCP tool.


## The A2A protocol — primitives you need to know

The spec defines four data shapes and four endpoints. That's it.

### Data shapes

**1. `AgentCard`** — the agent's public business card, served at `/.well-known/agent.json`. Describes who the agent is, what it can do, where to send tasks. Discovery starts here.

**2. `Task`** — the unit of delegated work. Has a `task_id`, a lifecycle state (`submitted → working → input-required | completed | failed | canceled`), an initiating `Message` from the client, the list of `Message`s the agent has produced, and zero or more final `Artifact`s.

**3. `Message`** — a single conversational turn. Composed of typed `Part`s: `TextPart`, `DataPart` (structured JSON), `FilePart` (URL or base64). Messages flow both directions during a task.

**4. `Artifact`** — the typed *deliverable* of the task. A research brief, a critique report, an audio file — whatever the agent was asked to produce. Lives on the terminal task object so clients can pull it after completion.

### Endpoints (the wire)

| Method | Path | Purpose |
|---|---|---|
| `GET`  | `/.well-known/agent.json` | Return the `AgentCard` — discovery |
| `POST` | `/tasks/send`              | Create a `Task` from an initial `Message`; returns the task in `submitted` or `working` state |
| `GET`  | `/tasks/{id}`              | Poll the task — get current state, accumulated messages, and (if terminal) the `Artifact`s |
| `GET`  | `/tasks/{id}/stream`       | Server-Sent Events stream of incremental `Message` and state updates (the L19 streaming pattern) |

That's the whole surface. Real production deployments add auth headers, signed JWT agent cards, push-notification webhooks for long tasks, and capability negotiation — but the core is the four endpoints above.


## 1 · Setup — install + load API key


In [ ]:
# Colab-friendly install. Run once per kernel.
!pip install -q anthropic fastapi uvicorn httpx nest_asyncio pydantic
import os, json, time, uuid, asyncio, threading
from enum import Enum
from typing import Optional, Literal, Any, Callable
from datetime import datetime, timezone

# Colab Secrets — set ANTHROPIC_API_KEY under the 🔑 sidebar before running.
try:
    from google.colab import userdata  # type: ignore
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    # Local Jupyter — assume the env var is already exported.
    assert os.environ.get('ANTHROPIC_API_KEY'), 'Set ANTHROPIC_API_KEY before running.'

import nest_asyncio; nest_asyncio.apply()  # so we can run uvicorn in a Colab thread
print('Setup OK.')


## 2 · `AgentCard` — the agent's business card

An AgentCard is just JSON. The spec includes ~20 fields; we'll model the essentials. The card is what makes A2A *discoverable* — point any A2A client at the agent's base URL and it can fetch the card to learn skills, endpoints, auth, and supported input/output modes.


In [ ]:
from pydantic import BaseModel, Field

class AgentSkill(BaseModel):
    """A single capability the agent advertises."""
    id: str
    name: str
    description: str
    examples: list[str] = Field(default_factory=list)
    tags: list[str] = Field(default_factory=list)

class AgentProvider(BaseModel):
    organization: str
    url: str

class AgentCard(BaseModel):
    """Public metadata an A2A agent serves at /.well-known/agent.json."""
    name: str
    description: str
    url: str                     # base URL — endpoints are url + /tasks/send etc.
    version: str = '1.0.0'
    provider: Optional[AgentProvider] = None
    default_input_modes: list[str] = Field(default_factory=lambda: ['text'])
    default_output_modes: list[str] = Field(default_factory=lambda: ['text', 'data'])
    capabilities: dict[str, bool] = Field(
        default_factory=lambda: {'streaming': True, 'push_notifications': False},
    )
    skills: list[AgentSkill] = Field(default_factory=list)

# Two cards — one per agent we'll spin up.
RESEARCHER_CARD = AgentCard(
    name='AutoResearcher.Researcher',
    description='Drafts a 5-bullet research brief on any topic.',
    url='http://127.0.0.1:8001',
    provider=AgentProvider(organization='LearnAI', url='https://example.com'),
    skills=[
        AgentSkill(
            id='draft_brief',
            name='Draft Research Brief',
            description='Given a topic, produce a 5-bullet evidence-flavored brief.',
            examples=['Topic: vector databases beyond pgvector'],
            tags=['research', 'writing'],
        ),
    ],
)

CRITIC_CARD = AgentCard(
    name='AutoResearcher.Critic',
    description='Critiques a research brief for hallucination / weak claims.',
    url='http://127.0.0.1:8002',
    provider=AgentProvider(organization='LearnAI', url='https://example.com'),
    skills=[
        AgentSkill(
            id='critique_brief',
            name='Critique Brief',
            description='Score a brief on factuality, specificity, and citation hygiene; emit issues.',
            examples=['Brief: …'],
            tags=['evaluation', 'critic'],
        ),
    ],
)

print(json.dumps(RESEARCHER_CARD.model_dump(), indent=2))


## 3 · `Task`, `Message`, `Artifact` — the lifecycle types

Mental picture: the **Task** is the conversation envelope; **Messages** flow inside it; one or more **Artifacts** are the typed output at the end.

The state machine:

```
  submitted ──► working ──┬──► completed         (terminal — happy path)
                          ├──► input-required    (paused — agent needs more info from client)
                          ├──► failed            (terminal — error)
                          └──► canceled          (terminal — client gave up)
```

`input-required` is the A2A version of the Lesson 11b *human-in-the-loop interrupt* — except here it's *another agent* (or a human client) being asked for more info, not a hard-coded checkpoint inside one process.


In [ ]:
class TaskState(str, Enum):
    SUBMITTED = 'submitted'
    WORKING = 'working'
    INPUT_REQUIRED = 'input-required'
    COMPLETED = 'completed'
    FAILED = 'failed'
    CANCELED = 'canceled'

TERMINAL = {TaskState.COMPLETED, TaskState.FAILED, TaskState.CANCELED}

class TextPart(BaseModel):
    type: Literal['text'] = 'text'
    text: str

class DataPart(BaseModel):
    type: Literal['data'] = 'data'
    data: dict  # arbitrary structured JSON

class FilePart(BaseModel):
    type: Literal['file'] = 'file'
    name: str
    mime_type: str
    uri: Optional[str] = None        # external link, OR
    bytes_b64: Optional[str] = None  # inline base64

Part = TextPart | DataPart | FilePart

class Message(BaseModel):
    role: Literal['user', 'agent']
    parts: list[Part]
    timestamp: str = Field(default_factory=lambda: datetime.now(timezone.utc).isoformat())

class Artifact(BaseModel):
    name: str
    description: Optional[str] = None
    parts: list[Part]
    index: int = 0

class TaskStatus(BaseModel):
    state: TaskState
    message: Optional[Message] = None  # optional status message ('searching…', 'reviewing…')
    timestamp: str = Field(default_factory=lambda: datetime.now(timezone.utc).isoformat())

class Task(BaseModel):
    id: str
    session_id: Optional[str] = None
    status: TaskStatus
    history: list[Message] = Field(default_factory=list)
    artifacts: list[Artifact] = Field(default_factory=list)
    metadata: dict = Field(default_factory=dict)

class SendTaskRequest(BaseModel):
    id: Optional[str] = None         # client-supplied id (idempotency); server fills if absent
    session_id: Optional[str] = None
    message: Message                 # the initiating user/agent message
    accepted_output_modes: list[str] = Field(default_factory=lambda: ['text', 'data'])

# Smoke test: build one of each.
demo_msg = Message(role='user', parts=[TextPart(text='Topic: durable agent execution with Temporal')])
print(demo_msg.model_dump_json(indent=2))


## 4 · A reusable `A2AServer` base — the four endpoints

Every A2A agent serves the same four endpoints. Let's write that once as a `FastAPI` factory and hand it a single function — `handle_task(task, send_update) -> Artifact` — that contains the agent's actual brain. Each concrete agent (Researcher, Critic) just supplies that one callable.

**Two design choices worth flagging:**

- **In-memory `TaskStore`.** Production uses Redis / Postgres / S3. We use a `dict`. Same interface.
- **Background worker per task.** `POST /tasks/send` returns *immediately* with state=`working`; the actual LLM call runs in `asyncio.create_task(...)`. Polling and SSE both read the same task object. This is how A2A handles long tasks without HTTP timeouts.


In [ ]:
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import StreamingResponse, JSONResponse

# ----- in-memory task store (would be Redis in prod) -----
class TaskStore:
    def __init__(self):
        self._tasks: dict[str, Task] = {}
        # one asyncio.Queue per task for SSE subscribers
        self._streams: dict[str, asyncio.Queue] = {}
        self._lock = asyncio.Lock()

    async def create(self, task: Task) -> None:
        async with self._lock:
            self._tasks[task.id] = task
            self._streams[task.id] = asyncio.Queue()

    async def update(self, task_id: str, *, status: Optional[TaskStatus] = None,
                     append_history: Optional[Message] = None,
                     append_artifact: Optional[Artifact] = None) -> None:
        async with self._lock:
            t = self._tasks.get(task_id)
            if t is None: return
            if status is not None:        t.status = status
            if append_history is not None: t.history.append(append_history)
            if append_artifact is not None: t.artifacts.append(append_artifact)
        # publish to SSE subscribers
        evt = {'task_id': task_id,
               'state': self._tasks[task_id].status.state.value,
               'message': append_history.model_dump() if append_history else None,
               'artifact': append_artifact.model_dump() if append_artifact else None}
        q = self._streams.get(task_id)
        if q is not None:
            await q.put(evt)

    async def get(self, task_id: str) -> Optional[Task]:
        async with self._lock:
            return self._tasks.get(task_id)

    def stream(self, task_id: str) -> asyncio.Queue:
        return self._streams.setdefault(task_id, asyncio.Queue())

# ----- A2A server factory -----
def make_a2a_app(card: AgentCard,
                  handle_task: Callable[[Task, Callable], Any]) -> FastAPI:
    """Returns a FastAPI app exposing the 4 A2A endpoints.
    handle_task(task, send_update) is an async fn that mutates the task via send_update()."""
    app = FastAPI(title=card.name)
    store = TaskStore()

    @app.get('/.well-known/agent.json')
    def get_card():
        return card.model_dump()

    @app.post('/tasks/send')
    async def send(req: SendTaskRequest):
        task_id = req.id or str(uuid.uuid4())
        task = Task(
            id=task_id, session_id=req.session_id,
            status=TaskStatus(state=TaskState.SUBMITTED),
            history=[req.message],
        )
        await store.create(task)
        await store.update(task_id, status=TaskStatus(state=TaskState.WORKING))

        async def send_update(**kw):  # close over store + task_id
            await store.update(task_id, **kw)

        # fire-and-forget — handler runs in background
        async def run():
            try:
                await handle_task(task, send_update)
                cur = await store.get(task_id)
                if cur and cur.status.state not in TERMINAL:
                    await send_update(status=TaskStatus(state=TaskState.COMPLETED))
            except Exception as e:
                await send_update(status=TaskStatus(
                    state=TaskState.FAILED,
                    message=Message(role='agent', parts=[TextPart(text=f'Internal error: {e!r}')]),
                ))
        asyncio.create_task(run())
        return (await store.get(task_id)).model_dump()

    @app.get('/tasks/{task_id}')
    async def get_task(task_id: str):
        t = await store.get(task_id)
        if t is None: raise HTTPException(404, 'task not found')
        return t.model_dump()

    @app.get('/tasks/{task_id}/stream')
    async def stream(task_id: str):
        q = store.stream(task_id)
        async def gen():
            # initial snapshot
            t = await store.get(task_id)
            if t:
                yield f'event: snapshot\ndata: {t.model_dump_json()}\n\n'
            while True:
                try:
                    evt = await asyncio.wait_for(q.get(), timeout=30)
                except asyncio.TimeoutError:
                    yield 'event: ping\ndata: {}\n\n'  # keep-alive
                    continue
                yield f'event: update\ndata: {json.dumps(evt)}\n\n'
                if evt.get('state') in {s.value for s in TERMINAL}:
                    yield 'event: done\ndata: {}\n\n'
                    return
        return StreamingResponse(gen(), media_type='text/event-stream')
    return app

print('A2AServer factory ready.')


## 5 · Concrete agents — Researcher and Critic

Each agent supplies one async `handle_task` that:

1. Reads the incoming user `Message` from `task.history[0]`.
2. Pushes a `working` status update so SSE subscribers see progress.
3. Calls Claude.
4. Pushes the final agent `Message` *and* a typed `Artifact` *and* sets state=`completed`.

Note how the agent's actual *cognitive logic* is small. The protocol does the heavy lifting.


In [ ]:
from anthropic import AsyncAnthropic
client = AsyncAnthropic()

MODEL = 'claude-haiku-4-5-20251001'  # cheap+fast for the demo

RESEARCHER_SYS = (
    'You are a research analyst. Given a topic, produce EXACTLY 5 bullet points,\n'
    'each ≤25 words, each carrying one concrete claim. Use plain text — no markdown bullets.\n'
    'Prefix each line with "- ". Do not invent citations; use "[evidence: …]" tags instead.\n'
)

CRITIC_SYS = (
    'You are a strict research critic. Given a draft brief, return JSON with three keys:\n'
    '  score   (int 1–10),\n'
    '  issues  (list of {bullet_index:int, problem:str, severity:"low|med|high"}),\n'
    '  verdict (one of "ship", "revise", "reject").\n'
    'Be terse. JSON only — no prose around it.\n'
)

def _first_text(msg: Message) -> str:
    for p in msg.parts:
        if isinstance(p, TextPart): return p.text
        if getattr(p, 'type', None) == 'text': return p.text  # tolerant
    return ''

# ---------- Researcher ----------
async def researcher_handler(task: Task, send_update):
    topic = _first_text(task.history[0])
    await send_update(status=TaskStatus(
        state=TaskState.WORKING,
        message=Message(role='agent', parts=[TextPart(text=f'Drafting brief on: {topic}')]),
    ))
    resp = await client.messages.create(
        model=MODEL, max_tokens=400, system=RESEARCHER_SYS,
        messages=[{'role': 'user', 'content': topic}],
    )
    brief = resp.content[0].text.strip()
    agent_msg = Message(role='agent', parts=[TextPart(text=brief)])
    artifact = Artifact(
        name='research_brief',
        description=f'5-bullet brief on {topic}',
        parts=[TextPart(text=brief), DataPart(data={'topic': topic, 'model': MODEL})],
    )
    await send_update(append_history=agent_msg, append_artifact=artifact,
                      status=TaskStatus(state=TaskState.COMPLETED))

# ---------- Critic ----------
async def critic_handler(task: Task, send_update):
    draft = _first_text(task.history[0])
    await send_update(status=TaskStatus(
        state=TaskState.WORKING,
        message=Message(role='agent', parts=[TextPart(text='Reviewing brief…')]),
    ))
    resp = await client.messages.create(
        model=MODEL, max_tokens=500, system=CRITIC_SYS,
        messages=[{'role': 'user', 'content': f'Brief to critique:\n{draft}'}],
    )
    raw = resp.content[0].text.strip()
    # tolerant JSON parse — strip code fences if model added them
    if raw.startswith('```'):
        raw = raw.strip('`').split('\n', 1)[1].rsplit('```', 1)[0]
    try:
        critique = json.loads(raw)
    except json.JSONDecodeError:
        critique = {'score': 0, 'issues': [], 'verdict': 'reject',
                    'parse_error': raw[:200]}
    agent_msg = Message(role='agent', parts=[
        TextPart(text=f'verdict={critique.get("verdict")} score={critique.get("score")}'),
        DataPart(data=critique),
    ])
    artifact = Artifact(name='critique', description='Structured critique JSON',
                       parts=[DataPart(data=critique)])
    await send_update(append_history=agent_msg, append_artifact=artifact,
                      status=TaskStatus(state=TaskState.COMPLETED))

researcher_app = make_a2a_app(RESEARCHER_CARD, researcher_handler)
critic_app     = make_a2a_app(CRITIC_CARD,     critic_handler)
print('Both agent apps built.')


## 6 · Spin both servers up (Colab pattern)

We run uvicorn in background threads so the notebook can also act as a client in the same kernel. In a real deploy you'd `docker compose up` two services.


In [ ]:
import uvicorn

def _serve(app, port):
    cfg = uvicorn.Config(app, host='127.0.0.1', port=port, log_level='warning')
    server = uvicorn.Server(cfg)
    asyncio.new_event_loop().run_until_complete(server.serve())

# idempotent — don't double-launch if you re-run the cell
_threads: dict[int, threading.Thread] = globals().get('_threads', {})
for app, port in [(researcher_app, 8001), (critic_app, 8002)]:
    if port in _threads and _threads[port].is_alive():
        continue
    t = threading.Thread(target=_serve, args=(app, port), daemon=True)
    t.start(); _threads[port] = t

time.sleep(2)  # let them bind
import httpx
print('Researcher card:', httpx.get('http://127.0.0.1:8001/.well-known/agent.json').json()['name'])
print('Critic card:    ', httpx.get('http://127.0.0.1:8002/.well-known/agent.json').json()['name'])


## 7 · `A2AClient` — discover, send, poll

Real life: clients are written *once* (against the protocol), and they work with any A2A agent. That's the payoff. Below is a ~30-line client that handles the full polling lifecycle.


In [ ]:
class A2AClient:
    def __init__(self, base_url: str, http: Optional[httpx.AsyncClient] = None,
                 poll_interval_s: float = 0.4, timeout_s: float = 60.0):
        self.base_url = base_url.rstrip('/')
        self._http = http or httpx.AsyncClient(timeout=timeout_s)
        self.poll_interval_s = poll_interval_s
        self.timeout_s = timeout_s
        self._card: Optional[AgentCard] = None

    async def discover(self) -> AgentCard:
        if self._card is None:
            r = await self._http.get(f'{self.base_url}/.well-known/agent.json')
            r.raise_for_status()
            self._card = AgentCard.model_validate(r.json())
        return self._card

    async def send_task(self, text: str, *, task_id: Optional[str] = None) -> Task:
        req = SendTaskRequest(
            id=task_id,
            message=Message(role='user', parts=[TextPart(text=text)]),
        )
        r = await self._http.post(f'{self.base_url}/tasks/send',
                                  json=req.model_dump(exclude_none=True))
        r.raise_for_status()
        return Task.model_validate(r.json())

    async def get_task(self, task_id: str) -> Task:
        r = await self._http.get(f'{self.base_url}/tasks/{task_id}')
        r.raise_for_status()
        return Task.model_validate(r.json())

    async def wait_until_done(self, task_id: str) -> Task:
        """Poll until terminal state or timeout."""
        t0 = time.monotonic()
        while True:
            t = await self.get_task(task_id)
            if t.status.state in TERMINAL:
                return t
            if time.monotonic() - t0 > self.timeout_s:
                raise TimeoutError(f'task {task_id} did not finish in {self.timeout_s}s')
            await asyncio.sleep(self.poll_interval_s)

print('A2AClient ready.')


## 8 · Live demo — Researcher → Critic over the wire

Three real HTTP round-trips:

1. Client → Researcher: discover card, send `Topic: …`, poll.
2. Pull the brief artifact off the completed Researcher task.
3. Client → Critic: send brief, poll, pull critique artifact.


In [ ]:
async def demo():
    researcher = A2AClient('http://127.0.0.1:8001')
    critic     = A2AClient('http://127.0.0.1:8002')

    rc = await researcher.discover(); print('🤝 discovered:', rc.name, '—', rc.skills[0].id)
    cc = await critic.discover();     print('🤝 discovered:', cc.name, '—', cc.skills[0].id)

    topic = 'Durable execution patterns for long-running LLM agents (Temporal vs Inngest)'
    t1 = await researcher.send_task(topic); print('\n📤 researcher task submitted:', t1.id)
    t1 = await researcher.wait_until_done(t1.id);  print('📥 researcher state:', t1.status.state.value)

    brief = _first_text(t1.history[-1])
    print('\n--- brief ---'); print(brief); print('-------------')

    t2 = await critic.send_task(brief); print('\n📤 critic task submitted:', t2.id)
    t2 = await critic.wait_until_done(t2.id); print('📥 critic state:', t2.status.state.value)

    crit_artifact = t2.artifacts[0]
    crit_json = next(p.data for p in crit_artifact.parts if isinstance(p, DataPart))
    print('\n--- critique ---'); print(json.dumps(crit_json, indent=2)); print('----------------')
    return t1, t2

t1, t2 = await demo()

# 💡 EXPERIMENT: change MODEL above to 'claude-sonnet-4-6' for one agent only.
# Now the two agents run on different model tiers — exactly the heterogeneity A2A unlocks.


## 9 · Streaming — consuming the critic's progress via SSE

L19 taught us SSE. A2A's `/tasks/{id}/stream` is that exact pattern: the server emits incremental `update` events as the agent works. Useful when the task is long and you want UI hints (`Reviewing…`, `Verifying citations…`).


In [ ]:
async def stream_demo(topic: str):
    researcher = A2AClient('http://127.0.0.1:8001')
    t = await researcher.send_task(topic)
    print(f'streaming task {t.id} …')
    async with httpx.AsyncClient(timeout=60) as h:
        async with h.stream('GET', f'http://127.0.0.1:8001/tasks/{t.id}/stream') as r:
            event_name = None
            async for line in r.aiter_lines():
                if line.startswith('event:'):
                    event_name = line.split(':', 1)[1].strip()
                elif line.startswith('data:'):
                    data = line.split(':', 1)[1].strip()
                    if event_name == 'done':
                        print('✅ done'); return
                    if event_name in ('snapshot', 'update'):
                        try:
                            d = json.loads(data)
                            label = d.get('state') or (d.get('status') or {}).get('state')
                            print(f'  ← [{event_name}] state={label}')
                        except json.JSONDecodeError:
                            pass

await stream_demo('Vector DB index families compared (HNSW vs IVF vs flat)')

# 💡 EXPERIMENT: add an extra send_update(...) call inside researcher_handler
# between the API call and the final completion — you'll see it appear here as an event.


## 10 · Porting AutoResearcher's critic-loop to A2A

Up through L31 the orchestrator called the critic in-process. Here we replace that call with an A2A round-trip. The orchestrator now doesn't know *or care* where the critic runs — could be another Python process, another container, another team's repo.

The loop is the same Constitutional pattern from L25: draft → critique → if `verdict==revise` then send the issues back as a follow-up message → repeat up to N rounds.


In [ ]:
async def autoresearcher_via_a2a(topic: str, *, max_rounds: int = 2) -> dict:
    researcher = A2AClient('http://127.0.0.1:8001')
    critic     = A2AClient('http://127.0.0.1:8002')

    history: list[dict] = []
    brief: str = ''
    crit: dict = {}

    # round 0 — initial draft
    rt = await researcher.send_task(topic)
    rt = await researcher.wait_until_done(rt.id)
    brief = _first_text(rt.history[-1])
    history.append({'round': 0, 'role': 'researcher', 'text': brief})

    for rnd in range(1, max_rounds + 1):
        ct = await critic.send_task(brief)
        ct = await critic.wait_until_done(ct.id)
        crit = next(p.data for p in ct.artifacts[0].parts if isinstance(p, DataPart))
        history.append({'round': rnd, 'role': 'critic', 'data': crit})
        if crit.get('verdict') == 'ship':
            break
        # ask researcher to revise — send the same topic with issues appended
        issues = '\n'.join(f"- bullet {i.get('bullet_index')}: {i.get('problem')}"
                            for i in crit.get('issues', []))
        revise_msg = f'Topic: {topic}\n\nPrevious draft:\n{brief}\n\nFix these issues:\n{issues}'
        rt = await researcher.send_task(revise_msg)
        rt = await researcher.wait_until_done(rt.id)
        brief = _first_text(rt.history[-1])
        history.append({'round': rnd, 'role': 'researcher', 'text': brief})

    return {'topic': topic, 'final_brief': brief, 'final_critique': crit, 'history': history}

result = await autoresearcher_via_a2a(
    'Calibration-aware refusal in production LLM agents', max_rounds=2)
print('rounds:', len(result['history']))
print('verdict:', result['final_critique'].get('verdict'))
print('--- final brief ---'); print(result['final_brief'])

# 💡 EXPERIMENT: take down the critic (set CRITIC_CARD url wrong / kill port 8002)
# and watch httpx error. THAT is the failure-isolation moment — orchestrator can fall
# back to a local critic, log it, and keep going. Compare to in-process: exception kills the run.


## 11 · When to use A2A vs MCP vs in-process

| Pattern | Lives where | Owns LLM? | Best for | Cost |
|---|---|---|---|---|
| **In-process function call** | Same Python process | n/a (just code) | Tight orchestrator loops; trivial helpers | None — but no isolation, no team boundaries |
| **MCP** (L14) | Separate process, *no LLM* | No — it's a tool server | Reusable typed tools (`search_web`, `read_file`) consumable by many agents | Network hop per call; schema versioning |
| **A2A** (today) | Separate process/repo/org, *owns LLM* | Yes — it's an agent | Heterogeneous agents (different models, different teams), failure isolation, capability discovery | Network hop, two LLM calls per round, schema versioning, harder debugging |

**Rule of thumb:** in-process until the boundary needs to *outlive a single repo*. Then MCP if it's a tool, A2A if it's a decision-maker.


## 12 · 12 pitfalls a Java engineer will recognize

These are exactly the distributed-system problems you've already met in microservices — A2A inherits them all.

1. **Idempotency.** Client retries a `POST /tasks/send` after a transient timeout — server creates a *second* task. Pass `req.id` from the client and treat duplicate IDs as the same task. Our store does the first part; production needs a dedup index.
2. **No authentication in the demo.** Production A2A: signed JWT in `Authorization: Bearer`, plus the `AgentCard` lists which auth scheme it accepts. Never expose `/tasks/send` openly.
3. **Long tasks block HTTP.** That's why `POST /tasks/send` returns immediately with `working` and the real work runs in a background coroutine. If you do the work synchronously you'll hit gateway timeouts at 30–60s.
4. **Polling thunder.** Hundreds of clients hitting `/tasks/{id}` at 200ms intervals melts the store. Prefer SSE (`/tasks/{id}/stream`) or push-notifications (capability flag in the card).
5. **Schema drift between agents.** Researcher emits `DataPart{topic,model}`; Critic also emits `DataPart` with `score,issues,verdict`. Two teams iterating independently *will* drift. Version your `Artifact.name` (`research_brief.v2`) and have clients tolerate unknown fields.
6. **Cross-process observability blackhole.** A bug in the critic shows up as a 5xx in the orchestrator's HTTP client. Propagate a `traceparent` header (W3C trace context) and aggregate spans in OpenTelemetry — L42 territory but plan for it now.
7. **Partial failure.** What if the researcher succeeds but the critic times out? Decide *upfront* whether the orchestrator returns the unreviewed draft, retries, or fails. Don't leave it to the exception handler.
8. **Streaming back-pressure.** Our SSE `Queue` is unbounded. A slow client + a chatty agent = memory bloat. Cap the queue or drop old events.
9. **AgentCard staleness.** Clients cache the card. When you add a new skill the cache lies. Honor `Cache-Control` and version the URL.
10. **Cost amplification.** Each agent owns its own LLM call. A 4-round loop = 8 LLM calls. Pipe both agents through the L22 `CostMeter` keyed by `task_id` to see the bill per task.
11. **Wrong primitive choice.** People reach for A2A for tools (`search_web`) — that's MCP. They reach for MCP for decisions (`triage_ticket`) — that's A2A. Re-read §11.
12. **`input-required` deadlocks.** Agent pauses waiting on the client; client doesn't know it should respond. Always emit a clear status `Message` *before* setting state=`input-required` so the client knows what to send next.


## 13 · Recap + what's next

**You built:**

- An `A2AServer` factory exposing the four standard endpoints with a background-task worker and SSE stream.
- Two concrete agents (Researcher / Critic) each ~30 lines of cognitive logic.
- An `A2AClient` that discovers an agent by `AgentCard` and drives the full task lifecycle.
- A working port of AutoResearcher's critic loop from in-process to A2A — same loop, different topology.

**You internalized:**

- A2A primitives — `AgentCard`, `Task`, `Message/Part`, `Artifact`, `TaskState`.
- Why decision-makers belong in A2A and tools belong in MCP.
- The 12 distributed-systems pitfalls that come bundled with the protocol.

### 🧪 Try before next lesson

1. Add a third agent: `Verifier` — given a brief, returns a `DataPart` listing which claims are factually checkable. Wire it after the critic in `autoresearcher_via_a2a`.
2. Wrap the critic's `handle_task` with an L25 Constitutional self-critique loop *inside the critic process*. Now you have multi-level agents.
3. Add `traceparent` header propagation through the `A2AClient` and log it in the server — that's your trace ID for L42.

### Coming next — Lesson 33 · Blackboard Architectures

A2A gives us *point-to-point* agent messaging. The next pattern up is **shared state** — multiple specialist agents reading and writing a common scratchpad, with a controller deciding who runs next. We'll build a 4-agent blackboard for collaborative research and compare it to today's request/response model.
